# 09 — Real-Data Sensitivity of the Population Slopes (SPARC only)

**Project:** *Is the RAR Universal? Correlated-error-aware constraints on the scatter and acceleration scale of the radial acceleration relation in SPARC and independent HI samples*

Three sensitivity analyses of the Notebook 05 population slopes, on the **real** data, requested in review:

* **S1 — Σ_x floor.** SPARC gives no baryonic-component errors, so Σ_x = 0 in Notebooks 03–05. The proposal defines Σ_x = diag[f′(x)² σ_x²]; here σ_x = 0.1 dex in log g_bar (data field `sigx`, already in the likelihood) and the three one-at-a-time fits are repeated.
* **S2 — errors in the covariates.** M_bar ∝ D² and uses Υ = 0.5, while D and Υ are sampled in the likelihood. Here the covariates are recomputed from K draws of D (log-normal, catalogue error) and Υ_disk (0.1 dex) per galaxy, re-standardised with the original mean and SD, and the M_bar and f_gas fits are repeated for each draw (Σ₀ does not depend on D or Υ). The spread of the slopes over draws measures how much covariate error could move them.
* **S3 — slopes inside a correlated-noise model (stopgap).** The population model is fitted with the correlated-noise covariance included, J K J with each galaxy's A_i and ℓ_i **fixed** at the full Model D posterior medians, and D, i, Υ **fixed** at their Model D medians. Only log a₀, β, σ_int, τ and the slopes are sampled. This removes the cost of sampling nuisance and GP hyperparameters, at the price of not propagating their uncertainty. It answers whether the slopes survive once correlated noise is in the model. Because A_i itself correlates with mass, a slope that vanishes here may have been absorbed by the mass-dependent amplitude.

**Unchanged:** priors, covariates, decision rule (Notebook 05), forward map. Every model change is checked numerically against the original likelihood.

**Runtime (Colab CPU, rough).** S1: 3 fits at the Notebook 05 budget (~25–35 min). S2: 2 × K fits at a reduced budget (K = 5: ~20–30 min). S3: 4 fits (baseline + 3 covariates) with dense per-galaxy covariances (~30–90 min). Total roughly 1.5–2.5 h. Every fit is saved as soon as it finishes; an interrupted run resumes with `RAR_RESUME_FROM`.

---
**v4 (converged Model D).** Use this version after the 04L "all galaxies" run has finished. S1 and S2 do not use Model D, so they are reused from your earlier run; only **S3 is refitted** with the new Model D noise. In the RUN SETTINGS cell set `RAR_RESUME_FROM` to the earlier Sensitivity run folder (the one with all S1–S3 results, `20261106T081353Z`) and keep `RAR_RERUN_S3 = "1"`. Use a **GPU runtime** (Colab: Runtime → Change runtime type → T4 GPU); S3 takes about 4 hours there and is impractical on CPU. Pin the same Model D run as Notebook 08X with `RAR_D_RUN_ID`.

In [ ]:
# =====================================================================
# RUN SETTINGS
# =====================================================================
import os
os.environ["RAR_RESUME_FROM"] = ""      # earlier 09 run folder, e.g. ".../05_Results/Sensitivity/20261106T081353Z"
os.environ["RAR_RERUN_S3"]    = "1"     # "1": drop the old S3 fits and refit S3 with the Model D chosen below; "0": plain resume
os.environ["RAR_D_RUN_ID"]    = ""      # 04L run ID of the converged Model D ("" = newest 153-galaxy Model D run)


## 2. Environment and Configuration

In [ ]:
import os, sys, importlib.util, subprocess
N_CPU = os.cpu_count() or 1
os.environ["XLA_FLAGS"] = f"--xla_force_host_platform_device_count={max(1, min(N_CPU, 8))}"
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.isdir("/kaggle/working")
REQUIRED = {"jax": "jax", "numpyro": "numpyro", "xarray": "xarray", "h5netcdf": "h5netcdf", "h5py": "h5py",
            "scipy": "scipy", "matplotlib": "matplotlib", "pandas": "pandas", "mpmath": "mpmath"}
missing = [p for m, p in REQUIRED.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import re, json, time, glob, hashlib, logging, platform, warnings, resource, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd, scipy, scipy.stats as sst, xarray as xr
import matplotlib, matplotlib.pyplot as plt
import jax, jax.numpy as jnp
from jax.scipy.linalg import cholesky as jcholesky, solve_triangular
import numpyro, numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value
from numpyro.infer.util import log_density, unconstrain_fn
from jax.scipy.special import ndtr, ndtri
from numpyro.diagnostics import split_gelman_rubin, effective_sample_size
numpyro.enable_x64()
numpyro.set_host_device_count(max(1, min(N_CPU, 8)))

RUN_TIMESTAMP = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
            "jax": jax.__version__, "numpyro": numpyro.__version__, "xarray": xr.__version__, "matplotlib": matplotlib.__version__,
            "cpu_count": N_CPU, "jax_devices": len(jax.devices()), "jax_backend": jax.default_backend(), "x64": bool(jax.config.read("jax_enable_x64"))}
print(f"Run {RUN_ID} | Colab: {IN_COLAB}"); print(VERSIONS)

In [ ]:
# =====================================================================
# CONFIGURATION
# =====================================================================
DRIVE_MOUNT_POINT = Path("/content/drive"); DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
PROJECT_ROOT = Path(os.environ.get("RAR_PROJECT_ROOT", str(DRIVE_ROOT / "RAR_Universality_Research")))
if IN_KAGGLE and "RAR_PROJECT_ROOT" not in os.environ:          # optional: also runs unattended on Kaggle
    import shutil
    cands = [p.parent for p in Path("/kaggle/input").rglob("03_Processed_Data") if p.is_dir()]
    full = [c for c in cands if any((c / "05_Results" / "Posterior_Samples").glob("03_bayesian_full_modelC_posterior*.nc"))]
    if not (full or cands): raise FileNotFoundError("Add your RAR_Universality_Research dataset as a Kaggle input.")
    src = sorted(full or cands, key=lambda p: len(p.parts))[0]
    PROJECT_ROOT = Path("/kaggle/working/RAR_Universality_Research")
    if not PROJECT_ROOT.exists(): shutil.copytree(src, PROJECT_ROOT, ignore=shutil.ignore_patterns("01_Datasets"))
PROC = PROJECT_ROOT / "03_Processed_Data"
DIRS = {"processed": PROC, "models": PROJECT_ROOT / "04_Trained_Models", "tables": PROJECT_ROOT / "05_Results" / "Tables",
        "posterior": PROJECT_ROOT / "05_Results" / "Posterior_Samples", "logs": PROC / "logs"}           # read-only inputs
NB02 = {"points": PROC / "sparc_rar_points_baseline.csv", "galaxies": PROC / "sparc_rar_galaxies_baseline.csv", "npz": PROC / "sparc_rar_model_inputs.npz"}
AUDIT01 = PROC / "sparc_galaxy_audit.csv"            # Notebook 01: derived Mbar and gas fraction (documented fiducial definitions)
NB03_RUN_ID = os.environ.get("RAR_NB03_RUN_ID", "")
OUT_ROOT = PROJECT_ROOT / "05_Results" / "Sensitivity" / RUN_ID                 # new, versioned; never overwrites
NB04_DIR = PROJECT_ROOT / "05_Results" / "Diagnostics" / "04_GP_Debugged"          # read-only: full Model D posterior
NB05_DIR = PROJECT_ROOT / "05_Results" / "Population_Dependence"                   # read-only: real slopes
OUT = {k: OUT_ROOT / k for k in ("posterior", "tables", "figures", "diagnostics", "config", "logs")}

KPC_IN_M, KMS_IN_MS, LN10 = 3.0856775814913673e19, 1.0e3, float(np.log(10.0))
REF = {"a0": 1.20e-10, "beta": 0.5, "upsilon_disk": 0.5, "upsilon_bulge": 0.7}
UPS_SIGMA_DEX = {"D": 0.10, "E": 0.20}             # Model C uses the 0.10 dex width (verified against Notebook 03)

# ---------------- Population model (proposal Sec. 4.5) ----------------
COVARIATES = {   # name: (definition, source)
    "logMbar": ("log10 baryonic mass [Msun] = log10(1.33 M_HI + 0.5 L[3.6]); Notebook 01 fiducial definition", "sparc_galaxy_audit.csv"),
    "logSB0":  ("log10 central disk surface brightness at 3.6 um [Lsun/pc^2] (SPARC SBdisk)", "sparc_rar_galaxies_baseline.csv"),
    "fgas":    ("gas fraction 1.33 M_HI / Mbar (Notebook 01 fiducial definition)", "sparc_galaxy_audit.csv"),
}
POP_PARAMS = ["log10_a0", "beta", "log_sigma_int", "log_tau"]
# Slope priors (per 1 SD of the covariate). They are not specified by the proposal: weakly informative regularisation choices,
# with a x2 sensitivity test. The intercepts (values at the sample-mean property) keep Notebook 03's priors exactly.
SLOPE_PRIOR_SD = {"log10_a0": 0.25,      # dex of a0 per SD
                  "beta": 0.10,          # beta per SD
                  "log_sigma_int": 0.5,  # natural-log units per SD (x1.65 per SD at 1 sigma)
                  "log_tau": 0.5}
BETA_FLOOR = 0.01                         # beta_i <= floor is outside the RAR's domain (zero likelihood); reported if ever approached
FITS_ONE_AT_A_TIME = list(COVARIATES)     # primary analysis
RUN_JOINT_FIT = True                      # secondary analysis (all three covariates together)
# ---------------- Notebook 09 plan ----------------
SIGX_FLOOR = float(os.environ.get("RAR_SIGX", 0.1))             # S1: sigma_x in log10 g_bar [dex]
K_IMPUTE = int(os.environ.get("RAR_K_IMPUTE", 5))               # S2: covariate draws
IMPUTE_SEED = 20261030
S2_FIT = {"chains": int(os.environ.get("RAR_S2_CHAINS", 2)), "warmup": int(os.environ.get("RAR_S2_WARMUP", 500)), "samples": int(os.environ.get("RAR_S2_SAMPLES", 1000))}
S3_FIT = {"chains": int(os.environ.get("RAR_S3_CHAINS", 4)), "warmup": int(os.environ.get("RAR_S3_WARMUP", 500)), "samples": int(os.environ.get("RAR_S3_SAMPLES", 1000))}
RESUME_FROM = os.environ.get("RAR_RESUME_FROM", "")             # earlier 09 run folder to resume from ("" = fresh)
RUN_PRIOR_SENSITIVITY = False
FAMILY_ALPHA = 0.05                       # family-wise error rate over the declared test family (12 slopes)

# ---------------- Sampler (Notebook 03 settings; Model C converged with them) ----------------
SEED = 20261027
TARGET_ACCEPT, MAX_TREE_DEPTH = 0.90, 10
FIT = {"chains": int(os.environ.get("RAR_CHAINS", 4)), "warmup": int(os.environ.get("RAR_WARMUP", 1000)), "samples": int(os.environ.get("RAR_SAMPLES", 1500))}
RHAT_MAX, ESS_BULK_MIN, ESS_TAIL_MIN, DIVERGENCES_MAX, BFMI_MIN = 1.01, 400, 400, 0, 0.3
ILOO = {"post_draws": int(os.environ.get("RAR_ILOO_DRAWS", 300)), "mc": int(os.environ.get("RAR_ILOO_MC", 64))}
ALLOW_OVERWRITE = False
PROGRESS = not IN_KAGGLE
print("Project root:", PROJECT_ROOT); print("Outputs ->", OUT_ROOT)

## 3. Drive, Logging and Notebook 03 Inputs

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_ROOT}")
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
if OUT_ROOT.exists() and any(OUT_ROOT.iterdir()):
    raise FileExistsError(f"{OUT_ROOT} already exists and is not empty; refusing to overwrite")
for d in OUT.values():
    d.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT["logs"] / f"09_sensitivity_{RUN_ID}.log"
logging.raiseExceptions = False
logger = logging.getLogger("rar09"); logger.setLevel(logging.INFO); logger.handlers.clear(); logger.propagate = False
_fh = logging.FileHandler(LOG_PATH, encoding="utf-8"); _fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
_sh = logging.StreamHandler(sys.stdout); _sh.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(_fh); logger.addHandler(_sh)

CHECKS = []
def record_check(section, name, passed, detail="", severity="critical"):
    status = "PASS" if passed else ("FAIL" if severity == "critical" else "WARN")
    CHECKS.append({"section": section, "check": name, "status": status, "severity": severity, "detail": str(detail)})
    (logger.info if passed else (logger.error if severity == "critical" else logger.warning))(f"[{status}] {section} | {name} | {detail}")
    if status == "FAIL":
        raise RuntimeError(f"Critical check failed: {section} | {name} | {detail}")
    return bool(passed)

def safe_path(path: Path) -> Path:
    if path.exists() and not ALLOW_OVERWRITE:
        new = path.with_name(f"{path.stem}_{RUN_ID}{path.suffix}"); logger.info(f"{path.name} exists; writing {new.name}"); return new
    return path

def sha256_of(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()
print("Log:", LOG_PATH)

In [ ]:
cfgs = []
for p in sorted(DIRS["models"].glob("03_bayesian_sampler_config*.json")):
    try:
        c = json.loads(p.read_text()); cm = c.get("full", {}).get("completed_models", {})
        cfgs.append({"file": p.name, "run_id": c.get("run_id"), "created": c.get("created_utc"), "full_models": ",".join(cm), "cfg": c})
    except Exception as exc:
        logger.warning(f"Could not read {p.name}: {exc}")
cfg_table = pd.DataFrame(cfgs)
display(cfg_table.drop(columns="cfg") if len(cfg_table) else "no Notebook 03 configurations found")
cands = [c for c in cfgs if "C" in c["full_models"].split(",") and (not NB03_RUN_ID or c["run_id"] == NB03_RUN_ID)]
record_check("NB03", "a Notebook 03 run with a completed full Model C exists", bool(cands), f"{len(cands)} candidate runs")
nb03 = max(cands, key=lambda c: c["created"]); CFG03 = nb03["cfg"]; RUN03 = nb03["run_id"]
print(f"Using Notebook 03 run {RUN03} (config {nb03['file']})")

def run_file(folder, stem, ext):
    # File written by run RUN03: suffixed name if present, else the base name.
    p1 = folder / f"{stem}_{RUN03}{ext}"; p0 = folder / f"{stem}{ext}"
    return p1 if p1.is_file() else p0

NC03 = {}
for m, info in CFG03["full"]["completed_models"].items():
    p = Path(info["netcdf"])
    if not p.is_file():                                         # stored path may be absolute from another mount; fall back to folder
        p = DIRS["posterior"] / Path(info["netcdf"]).name
    record_check("NB03", f"Model {m} posterior file exists", p.is_file(), str(p))
    attrs = xr.open_dataset(p, group="posterior", engine="h5netcdf").attrs
    record_check("NB03", f"Model {m} posterior carries run ID {RUN03}", attrs.get("run_id") == RUN03, f"file run_id {attrs.get('run_id')}")
    st = info["stats"]
    record_check("NB03", f"Model {m} met all convergence criteria in Notebook 03", bool(st["converged_by_all_criteria"]),
                 f"max R-hat {st['max_r_hat_all']:.4f}, min ESS {st['min_ess_bulk_all']:.0f}, divergences {st['divergences']}",
                 severity="warning" if os.environ.get("RAR_TEST_ALLOW_UNCONVERGED_NB03") == "1" else "critical")   # override: local testing only
    NC03[m] = p
chk03_path = run_file(DIRS["processed"], "03_bayesian_integrity_checks", ".csv")
chk03 = pd.read_csv(chk03_path)
record_check("NB03", "Notebook 03 check register has no FAIL rows", int((chk03.status == "FAIL").sum()) == 0,
             f"{chk03_path.name}: {len(chk03)} checks, {int((chk03.status=='FAIL').sum())} FAIL, {int((chk03.status=='WARN').sum())} WARN")
print("Notebook 03 warnings:"); display(chk03[chk03.status == "WARN"][["section", "check", "detail"]])
comp03_path = run_file(DIRS["tables"], "03_bayesian_model_comparison", ".csv")
comp03 = pd.read_csv(comp03_path) if comp03_path.is_file() else pd.DataFrame()
if len(comp03):
    display(comp03[[c for c in comp03.columns if c in ("stage", "model", "elpd_loo_galaxy", "p_loo", "n_pareto_k_gt_0.7", "sigma_int_median", "tau_median")]])
    bad_k = comp03[(comp03.get("n_pareto_k_gt_0.7", 0) > 0.1 * comp03.get("n_galaxies", 1))]
    record_check("NB03", "Notebook 03 galaxy-level PSIS-LOO reliable (Pareto k > 0.7 in < 10% of galaxies)", bad_k.empty,
                 bad_k[["stage", "model", "n_pareto_k_gt_0.7"]].values.tolist() if not bad_k.empty else "ok", severity="warning")

# Shared priors must be identical to Notebook 03
P03 = CFG03["priors"]
same_priors = (P03["log10_a0"]["low"] == -11.5 and P03["log10_a0"]["high"] == -8.5 and P03["beta"]["low"] == 0.05 and P03["beta"]["high"] == 1.0
               and P03["sigma_int"]["scale"] == 0.1 and P03["tau"]["scale"] == 0.1 and P03["upsilon_disk"]["sigma_dex"] == UPS_SIGMA_DEX["D"]
               and P03["upsilon_disk"]["median"] == REF["upsilon_disk"] and P03["upsilon_bulge"]["median"] == REF["upsilon_bulge"]
               and P03["inclination"]["low"] == 1.0 and P03["inclination"]["high"] == 90.0)
record_check("NB03", "shared priors in this notebook equal Notebook 03's saved priors", same_priors, "log10_a0, beta, sigma_int, tau, Upsilon, inclination bounds")
record_check("NB03", "Sigma_x convention unchanged (0: no documented baryonic-component errors)", CFG03.get("sigma_x_log_gbar", None) == 0.0, "0.0")

In [ ]:
for k in ("points", "galaxies", "npz"):
    rec = CFG03["input_checksums"].get(NB02[k].name)
    record_check("Sample", f"{NB02[k].name} unchanged since Notebook 03", rec is not None and sha256_of(NB02[k]) == rec, "SHA-256 compared")
pts = pd.read_csv(NB02["points"]); gals = pd.read_csv(NB02["galaxies"])
npz = np.load(NB02["npz"])
p_sorted = pts.sort_values(["galaxy_index", "point_index"]).reset_index(drop=True)
gtab = gals.sort_values("galaxy_index").reset_index(drop=True)
G = len(gtab); gi = p_sorted.galaxy_index.to_numpy(np.int32)
R0, V0, eV0 = npz["R_kpc"], npz["Vobs_kms"], npz["errV_kms"]
Vg0, Vd0, Vb0 = npz["Vgas_kms"], npz["Vdisk_ups1_kms"], npz["Vbul_ups1_kms"]
D0, eD0 = gtab.distance_mpc.to_numpy(float), gtab.e_distance_mpc.to_numpy(float)
I0, eI0 = gtab.inc_deg.to_numpy(float), gtab.e_inc_deg.to_numpy(float)
has_bulge = p_sorted.groupby("galaxy_index").Vbul.apply(lambda s: bool((s > 0).any())).reindex(range(G)).to_numpy()

# Positivity rule exactly as Notebook 03 (same seed and draw count); E's wider prior is evaluated for information only
def positivity(sig_dex, seed, n):
    rng = np.random.default_rng(seed)
    ud = 10 ** (np.log10(REF["upsilon_disk"]) + sig_dex * rng.standard_normal(n))
    ub = 10 ** (np.log10(REF["upsilon_bulge"]) + sig_dex * rng.standard_normal(n))
    p = np.ones(len(R0))
    for k in np.where(Vg0 <= 0)[0]:
        p[k] = float(np.mean(ud * Vd0[k] ** 2 + ub * Vb0[k] ** 2 + Vg0[k] * abs(Vg0[k]) > 0))
    return p
pos = CFG03["positivity_rule"]
p_pos = positivity(UPS_SIGMA_DEX["D"], CFG03["seed"], pos["mc_draws"])
gbar_ref = (Vg0 * np.abs(Vg0) + 0.5 * Vd0 ** 2 + 0.7 * Vb0 ** 2)
include = (p_pos >= pos["P_prior_min"]) & (gbar_ref > 0)
excl = sorted(zip(p_sorted.galaxy[~include], p_sorted.point_index[~include].astype(int)))
record_check("Sample", f"point set reproduces Notebook 03 ({pos['n_points_included']} points)", int(include.sum()) == pos["n_points_included"], f"{int(include.sum())}")
record_check("Sample", "excluded points identical to Notebook 03", [list(map(lambda v: v if isinstance(v, str) else int(v), e)) for e in excl] == pos["excluded_points"], excl)


## 4. Covariates, Likelihood and Model (identical to Notebooks 05 and 06)

In [ ]:
aud = pd.read_csv(AUDIT01)
cov = gtab[["galaxy", "galaxy_index", "SBdisk0_Lsun_pc2"]].merge(aud[["galaxy", "Mbar_1e9Msun_derived_fid", "fgas_derived_fid", "MHI_1e9Msun", "L36_1e9Lsun"]], on="galaxy", how="left")
cov["logMbar"] = np.log10(cov.Mbar_1e9Msun_derived_fid * 1e9)
cov["logSB0"] = np.log10(cov.SBdisk0_Lsun_pc2)
cov["fgas"] = cov.fgas_derived_fid
recheck = np.allclose(cov.Mbar_1e9Msun_derived_fid, 1.33 * cov.MHI_1e9Msun + 0.5 * cov.L36_1e9Lsun, rtol=1e-6)
record_check("Covariates", "Mbar reproduces 1.33 M_HI + 0.5 L[3.6] (Notebook 01 definition)", recheck, "")
record_check("Covariates", "galaxy order equals the model-input order", list(cov.galaxy_index) == list(range(G)), f"{G} galaxies")
for c in COVARIATES:
    record_check("Covariates", f"{c} finite for all {G} galaxies", bool(np.isfinite(cov[c]).all()), f"{int((~np.isfinite(cov[c])).sum())} non-finite")
COV_MEAN = {c: float(cov[c].mean()) for c in COVARIATES}; COV_SD = {c: float(cov[c].std(ddof=0)) for c in COVARIATES}
Zall = np.column_stack([(cov[c] - COV_MEAN[c]) / COV_SD[c] for c in COVARIATES])
cov_corr = pd.DataFrame(np.corrcoef(Zall.T), index=list(COVARIATES), columns=list(COVARIATES))
vif = {c: float(np.linalg.inv(cov_corr.values)[k, k]) for k, c in enumerate(COVARIATES)}
print("Covariate summary (physical units):"); display(cov[list(COVARIATES)].describe().T)
print("Correlation matrix of the standardised covariates:"); display(cov_corr.round(3))
print("Variance inflation factors (joint fit):", {k: round(v, 2) for k, v in vif.items()})
if max(vif.values()) > 2.5:
    logger.warning("Covariates are strongly correlated: joint-fit slopes will be partly degenerate. The one-at-a-time fits are primary (declared).")
cov_table = cov[["galaxy", "logMbar", "logSB0", "fgas"]].assign(**{f"z_{c}": Zall[:, k] for k, c in enumerate(COVARIATES)})

In [ ]:
def accelerations(D, inc_deg, ups_d, ups_b, idx, R0, V0, eV0, Vg0, Vd0, Vb0, D0, I0, xp=jnp):
    sD = (D / D0)[idx]
    s_inc = (xp.sin(xp.deg2rad(I0)) / xp.sin(xp.deg2rad(inc_deg)))[idx]
    R_m = R0 * sD * KPC_IN_M
    Vobs, eV = V0 * s_inc, eV0 * s_inc
    vbar2 = (Vg0 * xp.abs(Vg0) + ups_d[idx] * Vd0 ** 2 + ups_b[idx] * Vb0 ** 2) * sD
    g_obs = (Vobs * KMS_IN_MS) ** 2 / R_m
    g_bar = vbar2 * KMS_IN_MS ** 2 / R_m
    sig_V = 2.0 * eV / (Vobs * LN10)
    return g_obs, g_bar, sig_V

def f_rar(x, log10_a0, beta):
    t = jnp.exp(beta * (x - log10_a0) * LN10)
    return x - jnp.log10(-jnp.expm1(-t))

def fprime_rar(x, log10_a0, beta):
    t = jnp.exp(beta * (x - log10_a0) * LN10)
    return 1.0 - beta * t * jnp.exp(-t) / (-jnp.expm1(-t))

def residual_terms(params, data, nuis):
    log10_a0, beta = params[0], params[1]
    D, inc, ud, ub = nuis
    g_obs, g_bar, sig_V = accelerations(D, inc, ud, ub, data["idx"], data["R0"], data["V0"], data["eV0"],
                                        data["Vg0"], data["Vd0"], data["Vb0"], data["D0"], data["I0"])
    pos = g_bar > 0
    x = jnp.log10(jnp.where(pos, g_bar, 1e-10)); y = jnp.log10(g_obs)
    mu = f_rar(x, log10_a0, beta)
    return y - mu, sig_V, x, y, mu, pos

def galaxy_loglik_C(params, data, nuis):
    # Notebook 03 Model C likelihood (Sherman-Morrison, delta_i marginalised).
    log10_a0, beta, sig_int, tau, s = params
    r, sig_V, x, _, _, pos = residual_terms(params, data, nuis)
    d = (s * sig_V) ** 2 + (fprime_rar(x, log10_a0, beta) * data["sigx"]) ** 2 + sig_int ** 2
    seg = lambda v: jax.ops.segment_sum(v, data["idx"], num_segments=data["G"])
    S, A, Q, L, n = seg(1 / d), seg(r / d), seg(r ** 2 / d), seg(jnp.log(d)), seg(jnp.ones_like(d))
    ll = -0.5 * (Q - tau ** 2 * A ** 2 / (1 + tau ** 2 * S) + L + jnp.log1p(tau ** 2 * S) + n * jnp.log(2 * jnp.pi))
    return ll, jnp.all(pos)

def build_data_C(mask_points):
    rows = np.where(np.asarray(mask_points, bool))[0]
    gal_ids = np.unique(gi[rows]); remap = -np.ones(G, int); remap[gal_ids] = np.arange(len(gal_ids))
    idx = remap[gi[rows]]; Gs = len(gal_ids); bulge_local = np.where(has_bulge[gal_ids])[0]
    return {"idx": jnp.asarray(idx), "G": Gs, "gal_ids": gal_ids, "rows": rows, "bulge_local": jnp.asarray(bulge_local), "n_bulge": len(bulge_local),
            **{k: jnp.asarray(v[rows]) for k, v in dict(R0=R0, V0=V0, eV0=eV0, Vg0=Vg0, Vd0=Vd0, Vb0=Vb0).items()},
            "D0": jnp.asarray(D0[gal_ids]), "eD0": jnp.asarray(eD0[gal_ids]), "I0": jnp.asarray(I0[gal_ids]), "eI0": jnp.asarray(eI0[gal_ids]),
            "sigx": jnp.zeros(len(rows))}

def galaxy_loglik_pop(a_g, b_g, s_g, t_g, data, nuis):
    # Model C likelihood with per-galaxy log10 a0, beta, sigma_int, tau (arrays of length G).
    D, inc, ud, ub = nuis; idx = data["idx"]
    g_obs, g_bar, sig_V = accelerations(D, inc, ud, ub, idx, data["R0"], data["V0"], data["eV0"], data["Vg0"], data["Vd0"], data["Vb0"], data["D0"], data["I0"])
    pos = g_bar > 0
    x = jnp.log10(jnp.where(pos, g_bar, 1e-10)); y = jnp.log10(g_obs)
    a_p, b_p = a_g[idx], b_g[idx]
    r = y - f_rar(x, a_p, b_p)
    d = sig_V ** 2 + (fprime_rar(x, a_p, b_p) * data["sigx"]) ** 2 + s_g[idx] ** 2
    seg = lambda v: jax.ops.segment_sum(v, idx, num_segments=data["G"])
    S, A, Q, L, n = seg(1 / d), seg(r / d), seg(r ** 2 / d), seg(jnp.log(d)), seg(jnp.ones_like(d))
    ll = -0.5 * (Q - t_g ** 2 * A ** 2 / (1 + t_g ** 2 * S) + L + jnp.log1p(t_g ** 2 * S) + n * jnp.log(2 * jnp.pi))
    return ll, jnp.all(pos)

include_mask = include
data_full = build_data_C(include_mask)
Z_full = jnp.asarray(Zall[data_full["gal_ids"]])
print(f"Data: {data_full['G']} galaxies, {len(data_full['rows'])} points")

In [ ]:
ds_post = xr.open_dataset(NC03["C"], group="posterior", engine="h5netcdf")
ds_ll = xr.open_dataset(NC03["C"], group="log_likelihood", engine="h5netcdf")
names_nc = [str(x) for x in ds_post.galaxy.values]; names_here = list(gtab.galaxy.to_numpy()[data_full["gal_ids"]])
record_check("Regression", "Notebook 03 Model C galaxy order equals this notebook's", names_nc == names_here, f"{len(names_nc)} galaxies")
bulge_pos = [names_here.index(str(n)) for n in ds_post.bulge_galaxy.values] if "bulge_galaxy" in ds_post.dims else []
def nuis_from_nc(ds, c, d):
    ub = np.full(data_full["G"], REF["upsilon_bulge"])
    if bulge_pos: ub[bulge_pos] = ds["ups_bulge"].values[c, d]
    return (jnp.asarray(ds["D"].values[c, d]), jnp.asarray(ds["inc"].values[c, d]), jnp.asarray(ds["ups_disk"].values[c, d]), jnp.asarray(ub))
rng = np.random.default_rng(SEED); nchain, ndraw = ds_post.sizes["chain"], ds_post.sizes["draw"]; dev_saved, dev_C = 0.0, 0.0
for _ in range(10):
    c, d = int(rng.integers(nchain)), int(rng.integers(ndraw))
    th = [float(ds_post[k].values[c, d]) for k in ("log10_a0", "beta", "sigma_int", "tau")]
    nuis = nuis_from_nc(ds_post, c, d); Gn = data_full["G"]
    ll_p, _ = galaxy_loglik_pop(*(jnp.full(Gn, v) for v in th), data_full, nuis)
    ll_c, _ = galaxy_loglik_C((*th, 1.0), data_full, nuis)
    dev_saved = max(dev_saved, float(np.max(np.abs(np.asarray(ll_p) - ds_ll["galaxy_loglik"].values[c, d]))))
    dev_C = max(dev_C, float(np.max(np.abs(np.asarray(ll_p) - np.asarray(ll_c)))))
record_check("Regression", "zero-slope likelihood = Notebook 03 saved Model C log-likelihoods (10 draws, all galaxies)", dev_saved < 1e-6, f"max |diff| = {dev_saved:.2e}")
record_check("Regression", "zero-slope likelihood = Notebook 03 galaxy_loglik_C", dev_C < 1e-8, f"max |diff| = {dev_C:.2e}")
REGRESSION = {"max_abs_diff_vs_saved_NB03": dev_saved, "max_abs_diff_vs_C": dev_C}

In [ ]:
PRIORS_SHARED = {"log10_a0": P03["log10_a0"], "beta": P03["beta"], "sigma_int": P03["sigma_int"], "tau": P03["tau"]}
def pop_model(data, cfg):
    Z = cfg["Z"]; K = Z.shape[1]; mult = cfg.get("slope_prior_mult", 1.0)
    log10_a0 = numpyro.sample("log10_a0", dist.Uniform(P03["log10_a0"]["low"], P03["log10_a0"]["high"]))
    beta = numpyro.sample("beta", dist.Uniform(P03["beta"]["low"], P03["beta"]["high"]))
    sig_int = numpyro.sample("sigma_int", dist.HalfNormal(P03["sigma_int"]["scale"]))
    tau = numpyro.sample("tau", dist.HalfNormal(P03["tau"]["scale"]))
    if K:
        s = {p: numpyro.sample(f"slope_{p}", dist.Normal(jnp.zeros(K), mult * SLOPE_PRIOR_SD[p]).to_event(1)) for p in POP_PARAMS}
        a_g = log10_a0 + Z @ s["log10_a0"]; b_g = beta + Z @ s["beta"]
        s_g = sig_int * jnp.exp(Z @ s["log_sigma_int"]); t_g = tau * jnp.exp(Z @ s["log_tau"])
    else:
        Gn = data["G"]; a_g, b_g, s_g, t_g = (jnp.full(Gn, log10_a0), jnp.full(Gn, beta), jnp.full(Gn, sig_int), jnp.full(Gn, tau))
    with numpyro.plate("galaxies", data["G"]):                       # Notebook 03 Model C nuisance sampling, verbatim
        zD = numpyro.sample("zD", dist.Normal(0, 1))
        inc = numpyro.sample("inc", dist.TruncatedNormal(data["I0"], data["eI0"], low=P03["inclination"]["low"], high=P03["inclination"]["high"]))
        zUd = numpyro.sample("zUd", dist.Normal(0, 1))
    D = numpyro.deterministic("D", data["D0"] * jnp.exp(zD * data["eD0"] / data["D0"]))
    ud = numpyro.deterministic("ups_disk", P03["upsilon_disk"]["median"] * 10 ** (P03["upsilon_disk"]["sigma_dex"] * zUd))
    ub = jnp.full(data["G"], REF["upsilon_bulge"])
    if data["n_bulge"] > 0:
        with numpyro.plate("bulge_galaxies", data["n_bulge"]):
            zUb = numpyro.sample("zUb", dist.Normal(0, 1))
        ub = ub.at[data["bulge_local"]].set(P03["upsilon_bulge"]["median"] * 10 ** (P03["upsilon_bulge"]["sigma_dex"] * zUb))
        numpyro.deterministic("ups_bulge", ub[data["bulge_local"]])
    numpyro.deterministic("beta_min_over_galaxies", jnp.min(b_g))
    ll, ok = galaxy_loglik_pop(a_g, b_g, s_g, t_g, data, (D, inc, ud, ub))
    numpyro.factor("loglik", jnp.where(ok & (jnp.min(b_g) > BETA_FLOOR), jnp.sum(ll), -jnp.inf))

def fit_cfg(name, covs, mult=1.0):
    k = [list(COVARIATES).index(c) for c in covs]
    return {"name": name, "covariates": list(covs), "Z": Z_full[:, k] if k else jnp.zeros((data_full["G"], 0)), "slope_prior_mult": mult}

C_MED = {k: float(np.median(ds_post[k].values)) for k in ("log10_a0", "beta", "sigma_int", "tau")}
def init_values(cfg):
    v = {**C_MED, "zD": jnp.zeros(data_full["G"]), "inc": jnp.clip(data_full["I0"], 2.0, 89.5), "zUd": jnp.zeros(data_full["G"])}
    if data_full["n_bulge"]: v["zUb"] = jnp.zeros(data_full["n_bulge"])
    for p in POP_PARAMS:
        if cfg["Z"].shape[1]: v[f"slope_{p}"] = jnp.zeros(cfg["Z"].shape[1])
    return v

from numpyro.infer.util import log_density
lp0, _ = log_density(pop_model, (data_full, fit_cfg("baseline", [])), {}, init_values(fit_cfg("baseline", [])))
record_check("Model", "baseline (no slopes) log density finite at the initial values", bool(np.isfinite(float(lp0))), f"{float(lp0):.2f}")
cfg_test = fit_cfg("joint", list(COVARIATES)); iv = init_values(cfg_test)
lpj, trj = log_density(pop_model, (data_full, cfg_test), {}, iv)
lpb, trb = log_density(pop_model, (data_full, fit_cfg("baseline", [])), {}, init_values(fit_cfg("baseline", [])))
record_check("Model", "zero slopes give the baseline likelihood exactly", abs(float(trj["loglik"]["fn"].log_factor) - float(trb["loglik"]["fn"].log_factor)) < 1e-9, "joint vs baseline loglik factor")

In [ ]:
def run_fit(cfg, seed):
    kernel = NUTS(pop_model, target_accept_prob=TARGET_ACCEPT, max_tree_depth=MAX_TREE_DEPTH, init_strategy=init_to_value(values=init_values(cfg)))
    method = "parallel" if len(jax.devices()) >= FIT["chains"] else "sequential"
    mcmc = MCMC(kernel, num_warmup=FIT["warmup"], num_samples=FIT["samples"], num_chains=FIT["chains"], chain_method=method, progress_bar=PROGRESS)
    t0 = time.time()
    mcmc.run(jax.random.PRNGKey(seed), data_full, cfg, extra_fields=("diverging", "num_steps", "energy", "accept_prob"))
    jax.block_until_ready(mcmc.get_samples())
    return mcmc, time.time() - t0

def _rank_norm(x):
    r = sst.rankdata(x.ravel()).reshape(x.shape); return sst.norm.ppf((r - 3 / 8) / (x.size + 1 / 4))
def rank_rhat(x):
    return max(float(split_gelman_rubin(_rank_norm(x)[..., None])[0]), float(split_gelman_rubin(_rank_norm(np.abs(x - np.median(x)))[..., None])[0]))
def bulk_ess(x): return float(effective_sample_size(_rank_norm(x)[..., None])[0])
def tail_ess(x):
    q05, q95 = np.quantile(x, [0.05, 0.95])
    return float(min(effective_sample_size((x <= q05).astype(float)[..., None])[0], effective_sample_size((x >= q95).astype(float)[..., None])[0]))
SAMPLED = ["log10_a0", "beta", "sigma_int", "tau", "zD", "inc", "zUd", "zUb"] + [f"slope_{p}" for p in POP_PARAMS]
def diagnostics(mcmc):
    smp = mcmc.get_samples(group_by_chain=True); rows = []
    for k in SAMPLED:
        if k not in smp: continue
        v = np.asarray(smp[k]); flat = v.reshape(v.shape[0], v.shape[1], -1)
        rh = [rank_rhat(flat[..., j]) for j in range(flat.shape[2])]; eb = [bulk_ess(flat[..., j]) for j in range(flat.shape[2])]
        et = [tail_ess(flat[..., j]) for j in range(flat.shape[2])]
        rows.append({"parameter": k, "n_elements": flat.shape[2], "max_r_hat": max(rh), "min_ess_bulk": min(eb), "min_ess_tail": min(et)})
    dg = pd.DataFrame(rows); ex = mcmc.get_extra_fields(group_by_chain=True)
    div = np.asarray(ex["diverging"]); steps = np.asarray(ex["num_steps"]); en = np.asarray(ex["energy"])
    st = {"divergences": int(div.sum()), "divergences_per_chain": div.sum(1).tolist(), "max_tree_depth_hits": int((steps >= 2 ** MAX_TREE_DEPTH - 1).sum()),
          "mean_accept_prob": float(np.asarray(ex["accept_prob"]).mean()), "bfmi_per_chain": [float(np.mean(np.diff(e) ** 2) / np.var(e)) for e in en],
          "max_r_hat_all": float(dg.max_r_hat.max()), "min_ess_bulk_all": float(dg.min_ess_bulk.min()), "min_ess_tail_all": float(dg.min_ess_tail.min())}
    st["converged_by_all_criteria"] = bool(st["max_r_hat_all"] < RHAT_MAX and st["min_ess_bulk_all"] >= ESS_BULK_MIN and st["min_ess_tail_all"] >= ESS_TAIL_MIN
                                           and st["divergences"] <= DIVERGENCES_MAX and st["max_tree_depth_hits"] == 0 and min(st["bfmi_per_chain"]) >= BFMI_MIN)
    return dg, st

def save_fit(mcmc, cfg, path):
    smp = mcmc.get_samples(group_by_chain=True); ex = mcmc.get_extra_fields(group_by_chain=True)
    C_, S_ = np.asarray(smp["log10_a0"]).shape; names = gtab.galaxy.to_numpy()[data_full["gal_ids"]].astype(str)
    post, coords = {}, {"chain": np.arange(C_), "draw": np.arange(S_), "galaxy": names, "covariate": cfg["covariates"]}
    if data_full["n_bulge"]: coords["bulge_galaxy"] = names[np.asarray(data_full["bulge_local"])]
    for k, v in smp.items():
        v = np.asarray(v)
        if v.ndim == 2: post[k] = (("chain", "draw"), v)
        elif k.startswith("slope_"): post[k] = (("chain", "draw", "covariate"), v)
        elif v.shape[2] == data_full["G"]: post[k] = (("chain", "draw", "galaxy"), v)
        elif v.shape[2] == data_full["n_bulge"]: post[k] = (("chain", "draw", "bulge_galaxy"), v)
    if not cfg["covariates"]: coords.pop("covariate")
    path = safe_path(path)
    attrs = {"run_id": RUN_ID, "fit": cfg["name"], "covariates": json.dumps(cfg["covariates"]), "slope_prior_mult": cfg["slope_prior_mult"],
             "covariate_mean": json.dumps(COV_MEAN), "covariate_sd": json.dumps(COV_SD), "notebook03_run": RUN03}
    xr.Dataset(post, coords=coords, attrs=attrs).to_netcdf(path, group="posterior", mode="w", engine="h5netcdf")
    xr.Dataset({k2: (("chain", "draw"), np.asarray(ex[k1])) for k1, k2 in (("diverging", "diverging"), ("num_steps", "n_steps"), ("energy", "energy"), ("accept_prob", "acceptance_rate"))},
               coords={"chain": np.arange(C_), "draw": np.arange(S_)}).to_netcdf(path, group="sample_stats", mode="a", engine="h5netcdf")
    return path

## 5. Inputs from Notebooks 04 and 05 (read-only)

In [ ]:
cands = sorted(NB04_DIR.glob("*/posterior/04d_gp_full_model_D_posterior*.nc"))
record_check("Inputs", "Notebook 04 full Model D posterior found", bool(cands), str(cands[-1]) if cands else f"none under {NB04_DIR}")
PIN_D = os.environ.get("RAR_D_RUN_ID", "")            # optional: pin the Notebook 04 run whose Model D posterior defines the mocks
if PIN_D: cands = [p for p in cands if p.parents[1].name == PIN_D]
D_NC = None
for p_ in reversed(cands):                           # latest run first; skip runs fitted to a different galaxy set (e.g. the 149-galaxy 04L run)
    _ds = xr.open_dataset(p_, group="posterior", engine="h5netcdf")
    if [str(n) for n in _ds.galaxy.values] == names_here: D_NC = p_; break
record_check("Inputs", "a Model D posterior fitted to this galaxy sample was found", D_NC is not None, str(D_NC))
dpost = xr.open_dataset(D_NC, group="posterior", engine="h5netcdf")
names_D = [str(n) for n in dpost.galaxy.values]
print("Model D posterior used:", D_NC.parents[1].name)
record_check("Inputs", "Model D galaxy order equals this notebook's", names_D == names_here, f"{len(names_D)} galaxies")
TRUTH_CORR = {k: float(np.median(dpost[k].values)) for k in ("log10_a0", "beta", "tau", "sigma_int")}
A_TRUE = np.median(dpost["A"].values.reshape(-1, len(names_D)), axis=0); ELL_TRUE = np.median(dpost["ell"].values.reshape(-1, len(names_D)), axis=0)
TRUTH_WHITE = {k: float(np.median(ds_post[k].values)) for k in ("log10_a0", "beta", "tau", "sigma_int")}
print("A1 truth (Model D medians):", {k: round(v, 4) for k, v in TRUTH_CORR.items()}, "| A_i median", round(float(np.median(A_TRUE)), 2), "km/s; l_i median", round(float(np.median(ELL_TRUE)), 2), "kpc")
print("A2/B truth (NB03 Model C medians):", {k: round(v, 4) for k, v in TRUTH_WHITE.items()})
corrA = {c: float(np.corrcoef(np.log(A_TRUE), Zall[data_full["gal_ids"], k])[0, 1]) for k, c in enumerate(COVARIATES)}
print("Correlation of ln A_i (Model D) with the covariates:", {k: round(v, 2) for k, v in corrA.items()})

runs5 = sorted(NB05_DIR.glob("*/tables/05_slopes_all_fits.csv"))
record_check("Inputs", "Notebook 05 slope table found", bool(runs5), str(runs5[-1]) if runs5 else f"none under {NB05_DIR}")
slopes5 = pd.read_csv(runs5[-1]); real = slopes5[slopes5.fit.isin([f"one_{c}" for c in COVARIATES])].copy()
cfg5 = json.loads((runs5[-1].parents[1] / "config" / "05_population_config.json").read_text())
record_check("Inputs", "Notebook 05 used the same slope priors and covariate standardisation",
             cfg5["slope_prior_sd_per_SD"] == SLOPE_PRIOR_SD and all(abs(cfg5["covariate_mean"][c] - COV_MEAN[c]) < 1e-9 and abs(cfg5["covariate_sd"][c] - COV_SD[c]) < 1e-9 for c in COVARIATES),
             runs5[-1].parents[1].name)
REAL_SLOPE = {(r.parameter, r.covariate): (r.slope_per_SD_median, r.slope_per_SD_sd) for r in real.itertuples()}
display(real[["parameter", "covariate", "slope_per_SD_median", "slope_per_SD_sd", "verdict_family_wise"]])

## 6. Shared Fitting and Summary Functions

`fit_any` runs NUTS on any model with the Notebook 05 sampler settings and a given budget. `slope_table` reports, for each slope, the median, SD, z = median/SD and the Notebook 05 decision (the (1 − α′) interval from the draws excludes 0 **and** |z| > 2.87), plus convergence of the global parameters. Every fit's summary is appended to `tables/09_sensitivity_slopes.csv` as soon as it finishes.

In [ ]:
N_FAMILY = len(POP_PARAMS) * len(FITS_ONE_AT_A_TIME); ALPHA_P = FAMILY_ALPHA / N_FAMILY; ZCRIT = float(sst.norm.ppf(1 - ALPHA_P / 2))
def fit_any(model, data, cfg, init, seed, budget):
    kernel = NUTS(model, target_accept_prob=TARGET_ACCEPT, max_tree_depth=MAX_TREE_DEPTH, init_strategy=init_to_value(values=init))
    method = "parallel" if len(jax.devices()) >= budget["chains"] else "sequential"
    mcmc = MCMC(kernel, num_warmup=budget["warmup"], num_samples=budget["samples"], num_chains=budget["chains"], chain_method=method, progress_bar=PROGRESS)
    t0 = time.time(); mcmc.run(jax.random.PRNGKey(seed), data, cfg, extra_fields=("diverging", "num_steps", "energy", "accept_prob"))
    jax.block_until_ready(mcmc.get_samples()); return mcmc, time.time() - t0

GLOBAL_SITES = ["log10_a0", "beta", "sigma_int", "tau"] + [f"slope_{p}" for p in POP_PARAMS]
def slope_table(mcmc, cfg, meta):
    smp_c = mcmc.get_samples(group_by_chain=True); smp = mcmc.get_samples(); ex = mcmc.get_extra_fields(group_by_chain=True)
    rh, eb = [], []
    for k in GLOBAL_SITES:
        if k not in smp_c: continue
        v = np.asarray(smp_c[k]); v = v.reshape(v.shape[0], v.shape[1], -1)
        for j in range(v.shape[2]): rh.append(rank_rhat(v[..., j])); eb.append(bulk_ess(v[..., j]))
    div = int(np.asarray(ex["diverging"]).sum())
    conv = bool(max(rh) < 1.01 and min(eb) >= 400 and div == 0)
    rows = []
    base = {**meta, "max_rhat_global": max(rh), "min_ess_bulk_global": min(eb), "divergences": div, "converged_global": conv,
            "log10_a0": float(np.median(np.asarray(smp["log10_a0"]))), "beta": float(np.median(np.asarray(smp["beta"]))),
            "sigma_int": float(np.median(np.asarray(smp["sigma_int"]))), "tau": float(np.median(np.asarray(smp["tau"])))}
    if not cfg["covariates"]: return [{**base, "parameter": "-", "covariate": "-"}]
    for p in POP_PARAMS:
        x = np.asarray(smp[f"slope_{p}"])[:, 0]; c = cfg["covariates"][0]
        med, sd = float(np.median(x)), float(x.std()); lo, hi = np.quantile(x, [ALPHA_P / 2, 1 - ALPHA_P / 2])
        rv, rsd = REAL_SLOPE.get((p, c), (np.nan, np.nan))
        rows.append({**base, "parameter": p, "covariate": c, "slope_median": med, "slope_sd": sd, "z": med / sd,
                     "supported": bool((lo > 0 or hi < 0) and abs(med / sd) > ZCRIT and conv),
                     "nb05_slope": rv, "nb05_sd": rsd, "shift_in_nb05_sd": (med - rv) / rsd if rsd else np.nan})
    return rows

RES_CSV = OUT["tables"] / "09_sensitivity_slopes.csv"
done = pd.DataFrame()
if RESUME_FROM:
    prev = Path(RESUME_FROM) / "tables" / "09_sensitivity_slopes.csv"
    record_check("Resume", "previous results file found", prev.is_file(), str(prev))
    done = pd.read_csv(prev)
    if os.environ.get("RAR_RERUN_S3", "0") == "1":
        n_s3 = int(done.fit_id.astype(str).str.startswith("S3_").sum()); done = done[~done.fit_id.astype(str).str.startswith("S3_")]
        logger.info(f"Resume: dropped {n_s3} old S3 rows; S3 will be refitted with Model D {D_NC.parents[1].name}")
    done.to_csv(RES_CSV, index=False)
def already(fit_id): return (not done.empty) and (done.fit_id == fit_id).any()
def record(rows):
    pd.DataFrame(rows).to_csv(RES_CSV, mode="a", header=not RES_CSV.exists(), index=False)

## 7. S1 — Σ_x Floor

`sigx` enters the likelihood as (f′(x) σ_x)² added to each point's variance (Notebook 03 Model C definition). A zero-σ_x copy must reproduce the original likelihood exactly; the floor copy must lower it.

In [ ]:
data_sx = dict(data_full); data_sx["sigx"] = jnp.full(len(data_full["rows"]), SIGX_FLOOR)
cfg_t = fit_cfg("one_logMbar", ["logMbar"]); iv = init_values(cfg_t)
l0, _ = log_density(pop_model, (data_full, cfg_t), {}, iv); l1, _ = log_density(pop_model, (data_sx, cfg_t), {}, iv)
data_sx0 = dict(data_full); data_sx0["sigx"] = jnp.zeros(len(data_full["rows"])); l00, _ = log_density(pop_model, (data_sx0, cfg_t), {}, iv)
record_check("S1", "sigx = 0 copy reproduces the original log density", abs(float(l00) - float(l0)) < 1e-9, f"{float(l00) - float(l0):.2e}")
record_check("S1", f"sigx = {SIGX_FLOOR} changes the log density", abs(float(l1) - float(l0)) > 1e-6, f"{float(l0):.2f} -> {float(l1):.2f}")
for j, c in enumerate(FITS_ONE_AT_A_TIME):
    fid = f"S1_sigx_{c}"
    if already(fid): continue
    cfg = fit_cfg(f"one_{c}", [c])
    mc, secs = fit_any(pop_model, data_sx, cfg, init_values(cfg), SEED + 9100 + j, FIT)
    record(slope_table(mc, cfg, {"fit_id": fid, "analysis": "S1_sigx_floor", "draw": -1, "seconds": secs}))
    jax.clear_caches(); import gc; gc.collect()
    logger.info(f"{fid} done in {secs / 60:.1f} min")

## 8. S2 — Errors in the Covariates (Multiple Imputation)

For each draw k: D_i ~ log-normal(D_cat, e_D/D) and Υ_disk,i ~ 0.5 × 10^(0.1 N(0,1)); M_bar = (1.33 M_HI + Υ_disk L_[3.6]) (D/D_cat)² (both terms scale as D²), f_gas = 1.33 M_HI / M_bar; both re-standardised with the **original** mean and SD. The fits use a reduced budget; the decision column is shown for information only.

In [ ]:
cov_g = cov.set_index("galaxy_index").loc[np.arange(G)]
D0_all, eD0_all = np.asarray(D0), np.asarray(eD0)
rng_i = np.random.default_rng(IMPUTE_SEED); Z_DRAWS = []
for k in range(K_IMPUTE):
    Dk = D0_all * np.exp(rng_i.standard_normal(G) * eD0_all / D0_all)
    Uk = REF["upsilon_disk"] * 10 ** (P03["upsilon_disk"]["sigma_dex"] * rng_i.standard_normal(G))
    Mk = (1.33 * cov_g.MHI_1e9Msun.values + Uk * cov_g.L36_1e9Lsun.values) * (Dk / D0_all) ** 2 * 1e9
    zk = {"logMbar": (np.log10(Mk) - COV_MEAN["logMbar"]) / COV_SD["logMbar"],
          "fgas": (1.33 * cov_g.MHI_1e9Msun.values * 1e9 / Mk - COV_MEAN["fgas"]) / COV_SD["fgas"]}
    Z_DRAWS.append(zk)
zchk = (np.log10((1.33 * cov_g.MHI_1e9Msun.values + REF["upsilon_disk"] * cov_g.L36_1e9Lsun.values) * 1e9) - COV_MEAN["logMbar"]) / COV_SD["logMbar"]
record_check("S2", "imputation formula with D = D_cat and Upsilon = 0.5 reproduces the logMbar covariate", bool(np.allclose(zchk, Zall[:, list(COVARIATES).index("logMbar")], atol=1e-9)), "")
dz = np.array([Z_DRAWS[k]["logMbar"] - Zall[:, 0] for k in range(K_IMPUTE)])
print(f"SD of the covariate perturbation (logMbar, standardised units): {dz.std():.3f}; reliability ~ {1 / (1 + dz.var()):.3f}")
for k, zk in enumerate(Z_DRAWS):
    for c in ("logMbar", "fgas"):
        fid = f"S2_impute_{c}_{k:02d}"
        if already(fid): continue
        cfg = fit_cfg(f"one_{c}", [c]); cfg["Z"] = jnp.asarray(zk[c][data_full["gal_ids"]])[:, None]
        mc, secs = fit_any(pop_model, data_full, cfg, init_values(cfg), IMPUTE_SEED + 10 * k + (c == "fgas"), S2_FIT)
        record(slope_table(mc, cfg, {"fit_id": fid, "analysis": "S2_covariate_imputation", "draw": k, "seconds": secs}))
        jax.clear_caches(); import gc; gc.collect()
        logger.info(f"{fid} done in {secs / 60:.1f} min")

## 9. S3 — Population Slopes Inside a Correlated-Noise Model (Fixed Hyperparameters and Nuisance)

For galaxy i with its points padded to a common length, B_i = Σ_V,i + J_i K_i J_i + σ_int,i² I and C_i = B_i + τ_i² 11ᵀ. The log-likelihood uses the Cholesky factor of B_i and the matrix determinant lemma for the τ term (padded entries contribute exactly zero). K_i is the squared-exponential kernel of Notebook 04 in velocity space with catalogue radii, J_i = diag[2/(V_cat ln 10)], and A_i, ℓ_i, D, i, Υ are the full Model D posterior medians.

**Check:** with A_i = 0 this likelihood must equal Notebook 05's (Sherman–Morrison) likelihood at the same nuisance values, to 1e-8.

In [ ]:
def med_g(name):
    v = np.asarray(dpost[name].values); return np.median(v.reshape(-1, v.shape[-1]), axis=0)
NUIS_D = (jnp.asarray(med_g("D")), jnp.asarray(med_g("inc")), jnp.asarray(med_g("ups_disk")), None)
ub_full = np.full(data_full["G"], REF["upsilon_bulge"])
if data_full["n_bulge"]: ub_full[np.asarray(data_full["bulge_local"])] = med_g("ups_bulge")
NUIS_D = (NUIS_D[0], NUIS_D[1], NUIS_D[2], jnp.asarray(ub_full))
go_, gb_, sv_ = accelerations(*NUIS_D, data_full["idx"], data_full["R0"], data_full["V0"], data_full["eV0"], data_full["Vg0"], data_full["Vd0"], data_full["Vb0"], data_full["D0"], data_full["I0"])
record_check("S3", "g_bar > 0 at every point for the Model D median nuisance", bool(np.all(np.asarray(gb_) > 0)), "")
idx_np = np.asarray(data_full["idx"]); Gn = data_full["G"]; GAL_PTS = [np.where(idx_np == g)[0] for g in range(Gn)]
NMAX = max(len(p) for p in GAL_PTS)
P = np.zeros((Gn, NMAX), int); MSK = np.zeros((Gn, NMAX), bool)
for g, p in enumerate(GAL_PTS): P[g, :len(p)] = p; MSK[g, :len(p)] = True
R_cat = np.asarray(data_full["R0"]); W_cat = 2.0 / (np.asarray(data_full["V0"]) * LN10)
def build_fixed(A_vec, ELL_vec):
    x = np.log10(np.asarray(gb_)); y = np.log10(np.asarray(go_)); sv = np.asarray(sv_)
    Bfix = np.zeros((Gn, NMAX, NMAX))
    for g, p in enumerate(GAL_PTS):
        r = R_cat[p]; K = A_vec[g] ** 2 * np.exp(-0.5 * ((r[:, None] - r[None, :]) / ELL_vec[g]) ** 2)
        n = len(p); Bfix[g, :n, :n] = W_cat[p][:, None] * W_cat[p][None, :] * K + np.diag(sv[p] ** 2)
        Bfix[g, n:, n:] = np.eye(NMAX - n)
    return {"x": jnp.asarray(np.where(MSK, x[P], 0.0)), "y": jnp.asarray(np.where(MSK, y[P], 0.0)), "M": jnp.asarray(MSK.astype(float)),
            "B": jnp.asarray(Bfix), "n": jnp.asarray(MSK.sum(1).astype(float)), "G": Gn}
def gp_fixed_loglik(a_g, b_g, s_g, t_g, F):
    M = F["M"]; r = (F["y"] - f_rar(F["x"], a_g[:, None], b_g[:, None])) * M
    B = F["B"] + (s_g[:, None, None] ** 2) * (M[:, :, None] * jnp.eye(M.shape[1])[None])
    L = jnp.linalg.cholesky(B)
    sol = lambda v: jax.scipy.linalg.cho_solve((L, True), v[..., None])[..., 0]
    Binv_r, Binv_1 = jax.vmap(lambda Li, ri, mi: (jax.scipy.linalg.cho_solve((Li, True), ri), jax.scipy.linalg.cho_solve((Li, True), mi)))(L, r, M)
    Q = jnp.sum(r * Binv_r, 1); A_ = jnp.sum(M * Binv_r, 1); S = jnp.sum(M * Binv_1, 1)
    logdet = 2 * jnp.sum(jnp.log(jnp.diagonal(L, axis1=1, axis2=2)), 1)
    return -0.5 * (Q - t_g ** 2 * A_ ** 2 / (1 + t_g ** 2 * S) + logdet + jnp.log1p(t_g ** 2 * S) + F["n"] * jnp.log(2 * jnp.pi))
def pop_model_gp_fixed(F, cfg):
    Z = cfg["Z"]; K = Z.shape[1]; mult = cfg.get("slope_prior_mult", 1.0)
    log10_a0 = numpyro.sample("log10_a0", dist.Uniform(P03["log10_a0"]["low"], P03["log10_a0"]["high"]))
    beta = numpyro.sample("beta", dist.Uniform(P03["beta"]["low"], P03["beta"]["high"]))
    sig_int = numpyro.sample("sigma_int", dist.HalfNormal(P03["sigma_int"]["scale"]))
    tau = numpyro.sample("tau", dist.HalfNormal(P03["tau"]["scale"]))
    if K:
        s = {p: numpyro.sample(f"slope_{p}", dist.Normal(jnp.zeros(K), mult * SLOPE_PRIOR_SD[p]).to_event(1)) for p in POP_PARAMS}
        a_g = log10_a0 + Z @ s["log10_a0"]; b_g = beta + Z @ s["beta"]; s_g = sig_int * jnp.exp(Z @ s["log_sigma_int"]); t_g = tau * jnp.exp(Z @ s["log_tau"])
    else:
        a_g, b_g, s_g, t_g = (jnp.full(F["G"], log10_a0), jnp.full(F["G"], beta), jnp.full(F["G"], sig_int), jnp.full(F["G"], tau))
    ll = gp_fixed_loglik(a_g, b_g, s_g, t_g, F)
    numpyro.factor("loglik", jnp.where(jnp.min(b_g) > BETA_FLOOR, jnp.sum(ll), -jnp.inf))

F_GP = build_fixed(A_TRUE, ELL_TRUE); F_0 = build_fixed(np.zeros(Gn), ELL_TRUE)
tv = {k: jnp.full(Gn, v) for k, v in C_MED.items()}
ll_dense0 = gp_fixed_loglik(tv["log10_a0"], tv["beta"], tv["sigma_int"], tv["tau"], F_0)
ll_sm, _ = galaxy_loglik_pop(tv["log10_a0"], tv["beta"], tv["sigma_int"], tv["tau"], data_full, NUIS_D)
diff = float(jnp.max(jnp.abs(ll_dense0 - ll_sm)))
record_check("S3", "dense likelihood with A = 0 equals the Notebook 05 likelihood at the same nuisance", diff < 1e-8, f"max |diff| {diff:.2e}")
ll_gp = float(jnp.sum(gp_fixed_loglik(tv["log10_a0"], tv["beta"], tv["sigma_int"], tv["tau"], F_GP)))
record_check("S3", "log-likelihood with the Model D correlated noise is finite", bool(np.isfinite(ll_gp)), f"{ll_gp:.1f} vs {float(jnp.sum(ll_sm)):.1f} without")
S3_PLAN = [("baseline", [])] + [(f"one_{c}", [c]) for c in FITS_ONE_AT_A_TIME]
for j, (name, covs) in enumerate(S3_PLAN):
    fid = f"S3_gpfixed_{name}"
    if already(fid): continue
    cfg = fit_cfg(name, covs)
    init = {k: v for k, v in init_values(cfg).items() if k in ("log10_a0", "beta", "sigma_int", "tau") or k.startswith("slope_")}
    init["sigma_int"] = 0.01; init["tau"] = 0.05
    mc, secs = fit_any(pop_model_gp_fixed, F_GP, cfg, init, SEED + 9300 + j, S3_FIT)
    record(slope_table(mc, cfg, {"fit_id": fid, "analysis": "S3_slopes_inside_fixed_GP", "draw": -1, "seconds": secs}))
    jax.clear_caches(); import gc; gc.collect()
    logger.info(f"{fid} done in {secs / 60:.1f} min")

## 10. Summary, Figures and Report

In [ ]:
res = pd.read_csv(RES_CSV)
sl = res[res.parameter != "-"].copy()
s1 = sl[sl.analysis == "S1_sigx_floor"]; s3 = sl[sl.analysis == "S3_slopes_inside_fixed_GP"]
s2 = sl[sl.analysis == "S2_covariate_imputation"].groupby(["parameter", "covariate"]).agg(
    n_draws=("draw", "nunique"), slope_mean=("slope_median", "mean"), slope_sd_over_draws=("slope_median", "std"), mean_posterior_sd=("slope_sd", "mean"),
    nb05_slope=("nb05_slope", "first"), nb05_sd=("nb05_sd", "first"), mean_shift_in_nb05_sd=("shift_in_nb05_sd", "mean"),
    frac_supported=("supported", "mean")).reset_index()
cols = ["parameter", "covariate", "nb05_slope", "nb05_sd", "slope_median", "slope_sd", "z", "supported", "shift_in_nb05_sd", "converged_global", "max_rhat_global", "min_ess_bulk_global", "divergences"]
print("S1 (sigma_x floor):"); display(s1[cols].round(4))
print("S2 (covariate imputation; summary over draws):"); display(s2.round(4))
print("S3 (slopes inside the fixed correlated-noise model):"); display(s3[cols].round(4))
base3 = res[res.fit_id == "S3_gpfixed_baseline"]
if len(base3): print("S3 baseline globals:", base3[["log10_a0", "beta", "sigma_int", "tau", "converged_global", "max_rhat_global"]].round(4).to_dict("records"))
s1.to_csv(OUT["tables"] / "09_S1_sigx_floor.csv", index=False); s2.to_csv(OUT["tables"] / "09_S2_covariate_imputation.csv", index=False)
s3.to_csv(OUT["tables"] / "09_S3_slopes_inside_fixed_gp.csv", index=False)

def savefig(fig, name):
    p = safe_path(OUT["figures"] / name); fig.savefig(p, dpi=130, bbox_inches="tight"); plt.show(); return p
keys = [(p, c) for c in FITS_ONE_AT_A_TIME for p in POP_PARAMS]
fig, ax = plt.subplots(figsize=(8, 6)); y = np.arange(len(keys))
ax.errorbar([REAL_SLOPE[k][0] for k in keys], y, xerr=[REAL_SLOPE[k][1] for k in keys], fmt="ks", label="Notebook 05", ms=4)
for dfx, lab, off, mk in ((s1, f"S1 sigma_x = {SIGX_FLOOR}", -0.2, "o"), (s3, "S3 inside fixed GP", 0.2, "D")):
    t = pd.DataFrame(keys, columns=["parameter", "covariate"]).merge(dfx, on=["parameter", "covariate"], how="left")
    ax.errorbar(t.slope_median, y + off, xerr=t.slope_sd, fmt=mk, label=lab, ms=4)
t = pd.DataFrame(keys, columns=["parameter", "covariate"]).merge(s2, on=["parameter", "covariate"], how="left")
ax.errorbar(t.slope_mean, y + 0.1, xerr=np.sqrt(t.slope_sd_over_draws.fillna(0) ** 2 + t.mean_posterior_sd ** 2), fmt="^", label="S2 imputed covariates", ms=4)
ax.axvline(0, c="k", lw=0.6); ax.set_yticks(y); ax.set_yticklabels([f"{p} vs {c}" for p, c in keys], fontsize=8); ax.set_xlabel("slope per SD"); ax.legend(fontsize=7)
fig.tight_layout(); savefig(fig, "09_sensitivity_slopes.png")

pd.DataFrame(CHECKS).to_csv(OUT["diagnostics"] / "09_integrity_checks.csv", index=False)
config = {"run_id": RUN_ID, "created_utc": RUN_TIMESTAMP, "versions": VERSIONS, "notebook03_run_id": RUN03, "notebook04_D_posterior": str(D_NC),
          "notebook05_slopes": str(runs5[-1]), "sigx_floor": SIGX_FLOOR, "k_impute": K_IMPUTE, "impute_seed": IMPUTE_SEED, "S2_fit": S2_FIT, "S3_fit": S3_FIT,
          "notebook05_fit": FIT, "decision_rule": {"family": N_FAMILY, "per_test_alpha": ALPHA_P, "z_crit": ZCRIT}, "resume_from": RESUME_FROM}
(OUT["config"] / "09_sensitivity_config.json").write_text(json.dumps(config, indent=2, default=str))
pd.DataFrame([{"package": k, "version": str(v)} for k, v in VERSIONS.items()]).to_csv(OUT["config"] / "09_package_versions.csv", index=False)
chk = pd.DataFrame(CHECKS)
fmt = lambda d: "\n".join(f"* {r.parameter} vs {r.covariate}: {r.slope_median:+.4f} ± {r.slope_sd:.4f} (z {r.z:+.2f}; NB05 {r.nb05_slope:+.4f}; shift {r.shift_in_nb05_sd:+.2f} NB05 SD); "
                           f"{'supported' if r.supported else 'not supported'}; converged {r.converged_global} (R-hat {r.max_rhat_global:.3f})" for r in d.itertuples()) or "* (none)"
report = f"""# Notebook 09 — Real-Data Sensitivity of the Population Slopes — Report

*Run {RUN_ID} ({RUN_TIMESTAMP} UTC). Notebook 03 run {RUN03}; Notebook 04 Model D posterior {D_NC.parents[1].name}; Notebook 05 run {runs5[-1].parents[1].name}. This file exists only if the notebook ran.*

## S1 — Sigma_x floor ({SIGX_FLOOR} dex in log g_bar)
{fmt(s1)}

## S2 — covariate errors ({K_IMPUTE} draws of D and Upsilon_disk; logMbar and fgas fits)
{chr(10).join(f"* {r.parameter} vs {r.covariate}: mean {r.slope_mean:+.4f}, SD over draws {r.slope_sd_over_draws:.4f}, mean posterior SD {r.mean_posterior_sd:.4f}; NB05 {r.nb05_slope:+.4f}; mean shift {r.mean_shift_in_nb05_sd:+.2f} NB05 SD; supported in {r.frac_supported:.0%} of draws" for r in s2.itertuples()) or "* (none)"}

## S3 — slopes inside a correlated-noise model (A_i, l_i, D, i, Upsilon fixed at Model D medians)
{fmt(s3)}

## Interpretation rules (declared before running)
* S1/S2: a slope whose median moves by more than 1 NB05 SD, or whose support verdict changes, is sensitive to that assumption and must be reported as such.
* S3: slopes that remain beyond the Bonferroni cutoff with correlated noise in the model are candidate population dependences beyond correlated noise. Slopes that vanish may have been absorbed by the mass-dependent amplitude A_i (fixed here), so S3 cannot separate the two; it shows only whether the Model C slopes survive inside a correlated-noise model.
* S3 does not propagate nuisance or hyperparameter uncertainty; its intervals are too narrow by construction.

## Checks
{len(chk)} checks: {int((chk.status == "FAIL").sum())} FAIL, {int((chk.status == "WARN").sum())} WARN.
"""
rp = OUT["tables"] / "09_Sensitivity_Report.md"; rp.write_text(report, encoding="utf-8")
manifest = pd.DataFrame([{"file": str(p.relative_to(OUT_ROOT)), "bytes": p.stat().st_size} for p in sorted(OUT_ROOT.rglob("*")) if p.is_file()])
manifest.to_csv(OUT["config"] / "09_output_manifest.csv", index=False)
print(report); print("Outputs:", OUT_ROOT)